# Logistic Regression Assumptions — Tourism Practice Skeleton

**Short name (GitHub):** `LogReg_Tour`  
**Lab source:** Codecademy *Assumptions of Logistic Regression I & II* adapted to **hotel booking cancellations**.  
**Data:** `data/tour_hotels.csv` (740 bookings) and `data/tour_sessions.csv` (tour-package conversion practice).

Work this notebook first. Key: `LogReg_Tour_Solution.ipynb`. Helpers: `LogReg_Tour.py`. Flowchart: `logreg_tour_flowchart.png`.

> Revenue framing: **a missed cancellation (false negative) leaves an empty room**. Flagging a guest who will actually stay (false positive) only triggers a courtesy hold or a light overbook. Catch-rate of true cancels is the headline. Accuracy is secondary because most bookings are kept (467 stay / 273 cancel).


## Inline cheat-sheet (keep this cell visible)

See also **`LogReg_Tour_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Encode target | `df['cancelled'].map({'CANCEL':1,'STAY':0})` |
| Binary check | `value_counts()` — exactly two classes |
| Independence | `booking_id.nunique() == booking_id.count()` |
| Events-per-variable | `max_features = min(class counts) / 10` |
| Logit linearity | `sns.regplot(..., logistic=True)` looks sigmoidal |
| Multicollinearity | do not keep `booking_value` next to `nights` (it is ≈ nights × ADR) |
| Unregularized LR | `LogisticRegression(penalty=None, fit_intercept=True)` |
| Soft scores | `predict_proba(X)[:, 1]` = P(cancel) |
| Hard class at *t* | `(proba >= t).astype(int)` |
| Recall (cancels caught) | $\mathrm{TP}/(\mathrm{TP}+\mathrm{FN})$ |
| Precision | $\mathrm{TP}/(\mathrm{TP}+\mathrm{FP})$ |
| ROC / AUC | probabilities, not hard labels |
| Stratify | `train_test_split(..., stratify=y)` |
| Balanced weights | `class_weight='balanced'` |

Default $t=0.5$ is a library convention, not an overbooking policy. Lower *t* → more rooms held back, fewer surprise empties.


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_style("whitegrid")
np.set_printoptions(precision=4, suppress=True)
print("libraries ready")


## 1. Load and encode

### Task 1.1
Read `data/tour_hotels.csv`. Map `CANCEL→1`, `STAY→0`. Print `head()` and `cancelled.value_counts()`.


In [ ]:
# YOUR CODE HERE
df = None

print(df.head())
print(df.cancelled.value_counts())


## 2. Assumptions I

### Task 2.1 — binary target
Cancel is the **positive / minority** class.


In [ ]:
# YOUR CODE HERE
print(df.cancelled.value_counts())
print("n classes:", df.cancelled.nunique())


### Task 2.2 — independent bookings
Each row is one reservation. Test `booking_id.nunique() == booking_id.count()`.
Repeated stays by the same loyalty member would need a clustered / mixed logit later.


In [ ]:
# YOUR CODE HERE
unique_ids = None
print(unique_ids)


### Task 2.3 — events-per-variable cap
`max_features = min(class counts) / 10`. With 273 cancels the cap is 27.3. Five core features sit inside it.


In [ ]:
# YOUR CODE HERE
max_features = None
print(max_features)


### Task 2.4 — outlier screen
Plot the log-then-z-score boxplot of `cont_features`. Which column has the longest upper tail?


In [ ]:
all_features = [
    "lead_time", "nights", "adr", "booking_value", "party_size", "prev_cancels",
    "special_requests", "is_weekend", "online_channel", "has_deposit", "checkin_hour",
]
cont_features = [
    "lead_time", "nights", "adr", "booking_value", "party_size",
    "prev_cancels", "special_requests", "checkin_hour",
]
# YOUR CODE HERE


### Task 2.5 — drop extreme `special_requests`
Keep rows below the 99th percentile as `df_filtered`.


In [ ]:
# YOUR CODE HERE
q_hi = None
df_filtered = None
print("q99 =", q_hi, " kept =", len(df_filtered), " dropped =", len(df) - len(df_filtered))


### Task 2.6 — redraw the boxplot on `df_filtered`


In [ ]:
# YOUR CODE HERE


## 3. Assumptions II

### Task 3.1
Compare logistic `regplot` for `lead_time` (should look sigmoidal) and `checkin_hour` (flat / weak).


In [ ]:
# YOUR CODE HERE


### Task 3.2 — correlation heatmap
`booking_value` is essentially nights × ADR. Store a collinear pair as `correlated_pair` and do **not** put both size terms in the model.


In [ ]:
# YOUR CODE HERE
correlated_pair = []
print("correlated_pair:", correlated_pair)


## 4. scikit-learn implementation

Working set:

`core = ['lead_time','prev_cancels','online_channel','has_deposit','special_requests']`

### Task 4.1 — construct the estimator


In [ ]:
core = ["lead_time", "prev_cancels", "online_channel", "has_deposit", "special_requests"]
outcome = "cancelled"
x_train, x_test, y_train, y_test = train_test_split(
    df[core], df[outcome], random_state=0, test_size=0.3
)
# YOUR CODE HERE
log_reg = None
print(log_reg.get_params())


### Task 4.2 — fit and read coefficients
`has_deposit` and `special_requests` should come out **negative** (skin in the game, and guests who asked for extras tend to show). `prev_cancels` and `online_channel` should be positive.


In [ ]:
# YOUR CODE HERE
coefficients = None
intercept = None
print("coefficients:", coefficients)
print("intercept:", intercept)


### Task 4.3 — test-set metrics
Cancellation is noisy. Accuracy can look acceptable while recall at t=0.5 is weak — that is the point of the threshold section.


In [ ]:
# YOUR CODE HERE
y_pred = None
accuracy = precision = recall = f1 = None
print(accuracy, precision, recall, f1)


### Task 4.4 — confusion matrix
The expensive cell for a full hotel is **FN** (actual cancel, predicted stay → empty room).


In [ ]:
# YOUR CODE HERE
test_conf_matrix = None
print(test_conf_matrix)


## 5. Prediction thresholds

### Task 5.1
Rebuild the 0.5 class from `predict_proba` and confirm it matches `predict`.


In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
# YOUR CODE HERE
y_pred_class = None
diff = None
print("same as predict()?", diff)


### Task 5.2 / 5.3 — CMs at 25%, 50%, 75%
Lower *t* catches more cancels and flags more loyal guests (the overbooking / courtesy-hold trade).


In [ ]:
# YOUR CODE HERE
print("CM 50%"); print(None)
print("CM 25%"); print(None)
print("CM 75%"); print(None)


### Task 5.4 — revenue threshold
Sweep `thresh = np.linspace(0, 1, 100)`. What is the **lowest** threshold at which FN first reaches 12? Store as `thresh_choice`.


In [ ]:
thresh = np.linspace(0, 1, 100)
false_negatives = []
# YOUR CODE HERE
thresh_choice = None
print("thresh_choice =", thresh_choice)


## 6. ROC curve and AUC


In [ ]:
# YOUR CODE HERE — plot ROC vs DummyClassifier(most_frequent)


In [ ]:
# YOUR CODE HERE
roc_auc = None
print("ROC AUC:", roc_auc)


## 7. Class imbalance

Positivity ≈ 273/740 ≈ 0.37.

### Task 7.1 — stratified split (`random_state=6`)


In [ ]:
x_train_u, x_test_u, y_train_u, y_test_u = train_test_split(
    df[core], df[outcome], random_state=6, test_size=0.3
)
print("unstrat train pos", y_train_u.mean(), "test pos", y_test_u.mean())
# YOUR CODE HERE — x_train_str, x_test_str, y_train_str, y_test_str


### Task 7.2 — positivity rates after stratification

In [ ]:
# YOUR CODE HERE
str_train_positivity_rate = str_test_positivity_rate = None
print(str_train_positivity_rate, str_test_positivity_rate)


### Task 7.3 — recall / accuracy on the stratified fold

In [ ]:
# YOUR CODE HERE
recall_str = accuracy_str = None
print("stratified recall, acc:", recall_str, accuracy_str)


### Task 7.4 / 7.5 — `class_weight='balanced'` on the unstratified rs=6 split

In [ ]:
# YOUR CODE HERE
log_reg_bal = None
recall_bal = accuracy_bal = None
print("balanced recall, acc:", recall_bal, accuracy_bal)


## 8. Alternate code
### Task 8.1 — statsmodels Logit


In [ ]:
# YOUR CODE HERE


### Task 8.2 — scale-then-LR pipeline

In [ ]:
# YOUR CODE HERE


### Task 8.3 — NumPy threshold helper

In [ ]:
def predict_at(proba, t=0.5):
    # YOUR CODE HERE
    pass

for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    pred = predict_at(y_pred_prob[:, 1], t)
    cm = confusion_matrix(y_test, pred)
    print(t, "FN", cm[1, 0], "FP", cm[0, 1])


## 9. More practice

### Task 9.1 — add `is_weekend` and `nights`
Does AUC move enough to spend two more EPV slots?


In [ ]:
extra = core + ["is_weekend", "nights"]
# YOUR CODE HERE


### Task 9.2 — transfer problem: tour-package conversion
`data/tour_sessions.csv` — `page_views`, `session_min`, `promo`, `repeat_visitor`, `mobile` → `booked`.

1. Positivity rate and 10-EPV cap.
2. Drop `scroll_depth` if it is collinear with `session_min`.
3. Fit unregularized LR; print coefficients, recall, AUC.
4. Pick a threshold that keeps FN ≤ 8 on the test fold (or document that you cannot).


In [ ]:
sessions = pd.read_csv("data/tour_sessions.csv")
# YOUR CODE HERE


### Task 9.3 — which metric when?
One sentence each: 95% occupancy weekend, shoulder-season midweek, tourism-board quarterly pack.


In [ ]:
weekend = """..."""
shoulder = """..."""
board = """..."""
print(weekend); print(shoulder); print(board)


## 10. Simulation (edit the parameters)

Change `N`, `NOISE`, `T` and re-run.

* What happens to cancel-catch-rate if you raise `T` from 0.5 to 0.8?
* Does `class_weight='balanced'` still help at `N=150`?
* How much label noise (`NOISE=0.10`) drags AUC?


In [ ]:
# --- editable parameters ---
N = 740
N_REPS = 20
NOISE = 0.00
T = 0.50
TEST_SIZE = 0.30
SEED = 0
# ---------------------------
rng = np.random.default_rng(SEED)
rows_def, rows_bal = [], []
pool_X = df[core].to_numpy(); pool_y = df[outcome].to_numpy()
for r in range(N_REPS):
    idx = rng.choice(len(df), size=N, replace=(N > len(df)))
    X, y = pool_X[idx], pool_y[idx]
    try:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r, stratify=y)
    except ValueError:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy(); ytr[flip] = 1 - ytr[flip]
    for tag, cw in (("default", None), ("balanced", "balanced")):
        m = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, class_weight=cw)
        m.fit(Xtr, ytr)
        proba = m.predict_proba(Xte)[:, 1]
        pred = (proba >= T).astype(int)
        rec = dict(
            recall=recall_score(yte, pred, zero_division=0),
            accuracy=accuracy_score(yte, pred),
            auc=roc_auc_score(yte, proba) if len(np.unique(yte)) == 2 else np.nan,
        )
        (rows_def if tag == "default" else rows_bal).append(rec)
sim_def = pd.DataFrame(rows_def); sim_bal = pd.DataFrame(rows_bal)
print("DEFAULT\n", sim_def.agg(["mean", "std"]).round(3))
print("BALANCED\n", sim_bal.agg(["mean", "std"]).round(3))
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, col in zip(axes, ["recall", "accuracy", "auc"]):
    ax.boxplot([sim_def[col].dropna(), sim_bal[col].dropna()], labels=["default", "balanced"])
    ax.set_title(col); ax.set_ylim(0.30, 1.02)
fig.suptitle(f"N={N}  T={T}  noise={NOISE}  reps={N_REPS}", y=1.03)
plt.tight_layout(); plt.show()


## 11. Audience rewrite

Rewrite one finding four ways (expert revenue analyst / PMS technician / hotel GM / guest-facing staff). ≤ 80 words each.
Example finding: “at t=0.25 we miss 8 cancels and flag 102 stayers; AUC ≈ 0.68”.


In [ ]:
expert = """..."""
technician = """..."""
executive = """..."""
nonspecialist = """..."""
print(expert); print(technician); print(executive); print(nonspecialist)


## 12. Top-10 applications — and when **not** to use this


In [ ]:
applications = []
not_appropriate = []
for row in applications: print(row)
print("--- not appropriate ---")
for row in not_appropriate: print(row)


## 13. Done-when checklist

- [ ] Encoded target, unique booking_id, 10-EPV cap
- [ ] Outlier boxplots before / after the special-requests filter
- [ ] Two logistic regplots and a labelled heatmap
- [ ] 5-feature model, 4 metrics, CM
- [ ] Threshold CMs + `thresh_choice`
- [ ] ROC vs dummy + AUC
- [ ] Stratify + `class_weight='balanced'`
- [ ] One alternate stack
- [ ] Extra-feature and tour-session practice
- [ ] Simulation that moves when you edit `N` / `T` / `NOISE`
- [ ] Four audience paragraphs + applications list

**Companions:** solution notebook, reusable template, `LogReg_Tour.py`, cheat sheet, 1-page report, memo, strategy guide, `logreg_tour_flowchart.png`.
